In [1]:
import torch
import torch.nn as nn
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm

In [ ]:
import pickle
with open('weights/word2index_eng.pkl', 'rb') as f:
    word2index_eng = pickle.load(f)
with open('weights/word2index_fr.pkl', 'rb') as f:
    word2index_fr = pickle.load(f)

index2word_eng = {id:word for (word, id) in word2index_eng.items()}
index2word_fr = {id:word for (word, id) in word2index_fr.items()}

nlp_eng = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp_fr = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

In [3]:
word2index_eng

{'<PAD>': 0,
 '<UNK>': 1,
 '<SOS>': 2,
 '<EOS>': 3,
 'go': 4,
 'hi': 5,
 'run': 6,
 'who': 7,
 'wow': 8,
 'duck': 9,
 'fire': 10,
 'help': 11,
 'hide': 12,
 'jump': 13,
 'stop': 14,
 'wait': 15,
 'begin': 16,
 'on': 17,
 'hello': 18,
 'i': 19,
 'see': 20,
 'try': 21,
 'won': 22,
 'oh': 23,
 'no': 24,
 'relax': 25,
 'shoot': 26,
 'smile': 27,
 'sorry': 28,
 'attack': 29,
 'buy': 30,
 'it': 31,
 'cheers': 32,
 'eat': 33,
 'exhale': 34,
 'get': 35,
 'up': 36,
 'now': 37,
 'got': 38,
 'hop': 39,
 'in': 40,
 'hug': 41,
 'me': 42,
 'fell': 43,
 'fled': 44,
 'hunt': 45,
 'knit': 46,
 'know': 47,
 'left': 48,
 'lied': 49,
 'lost': 50,
 'paid': 51,
 'pass': 52,
 'quit': 53,
 'swim': 54,
 "'m": 55,
 '19': 56,
 'ok': 57,
 'inhale': 58,
 'listen': 59,
 'way': 60,
 'really': 61,
 'thanks': 62,
 'we': 63,
 'ask': 64,
 'tom': 65,
 'him': 66,
 'awesome': 67,
 'be': 68,
 'calm': 69,
 'cool': 70,
 'fair': 71,
 'kind': 72,
 'nice': 73,
 'beat': 74,
 'burn': 75,
 'bury': 76,
 'call': 77,
 'us': 78,
 'come

In [4]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, nlp_eng, nlp_fr, word2index_eng, word2index_fr, root_eng, root_fr):
        self.data = []
        doc_bin_eng = DocBin().from_disk(root_eng).get_docs(nlp_eng.vocab)
        doc_bin_fr = DocBin().from_disk(root_fr).get_docs(nlp_fr.vocab)
        i = 0
        for doc_eng, doc_fr in zip(doc_bin_eng, doc_bin_fr):
            i += 1
            if i > 240500:
                continue
            sentence_eng = [2]
            for token in doc_eng:
                if not token.is_punct and not token.is_space:
                    sentence_eng.append(word2index_eng[token.text.lower()])
            
            sentence_fr = [2]
            for token in doc_fr:
                if not token.is_punct and not token.is_space:
                    sentence_fr.append(word2index_fr[token.text.lower()])

            sentence_eng.append(3)
            sentence_fr.append(3)

            sentence_eng = self.padding(sentence_eng, 40)
            sentence_fr = self.padding(sentence_fr, 40)

            self.data.append((torch.tensor(sentence_eng), torch.tensor(sentence_fr)))
    
    def padding(self, vector, size):
        while len(vector) != size:
            vector.append(0)
        return vector
    
    def __getitem__(self, index):
        return self.data[index]
    def __len__(self):
        return len(self.data)

In [5]:
dataset = CustomDataset(nlp_eng, nlp_fr, word2index_eng, word2index_fr, 'eng_data.spacy', 'fr_data.spacy')
len(dataset)

240500

In [6]:
dataset[0]

(tensor([2, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 tensor([    2, 16720,     3,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0]))

In [19]:
dataloader = torch.utils.data.DataLoader(dataset, 32, True)

In [8]:
import torch
import torch.nn as nn

class Translator(nn.Module):
    def __init__(self, w2v_eng_root, w2v_fr_root, in_f, out_vocab_size, hid_f):
        super().__init__()
        self.embedding_eng = nn.Embedding.from_pretrained(torch.load(w2v_eng_root, weights_only=True), freeze=False)
        self.embedding_fr = nn.Embedding.from_pretrained(torch.load(w2v_fr_root, weights_only=True), freeze=False)
        self.encoder = nn.LSTM(input_size=in_f, hidden_size=hid_f, num_layers=2, batch_first=True)
        self.decoder = nn.LSTM(input_size=in_f, hidden_size=hid_f, num_layers=2, batch_first=True)
        self.linear = nn.Linear(in_features=hid_f, out_features=out_vocab_size)
        
        self.sos_idx = 2
    
    def forward(self, x, y=None, max_len=40):
        batch_size = x.shape[0]
        out_vocab_size = self.linear.out_features
        RATIO = 0.4
        # If y is provided (Training), use its length. If not (Inference), use max_len.
        trg_len = y.shape[1] if y is not None else max_len
        
        outputs = torch.zeros(batch_size, trg_len, out_vocab_size).to(x.device)
        
        # --- ENCODER ---
        x_emb = self.embedding_eng(x)
        _, (hn, cn) = self.encoder(x_emb) 
        
        # --- DECODER INIT ---
        # If we have y, grab the first column. If not, create a tensor full of the <SOS> index.
        if y is not None:
            dec_input = y[:, 0]
        else:
            dec_input = torch.full((batch_size,), self.sos_idx, dtype=torch.long, device=x.device)
        
        # --- AUTOREGRESSIVE LOOP ---
        for t in range(1, trg_len):
            
            dec_input_emb = self.embedding_fr(dec_input).unsqueeze(1)
            output, (hn, cn) = self.decoder(dec_input_emb, (hn, cn))
            prediction = self.linear(output.squeeze(1))
            outputs[:, t, :] = prediction
            
            # Pure Greedy Decoding
            top1 = prediction.argmax(1)

            rand = torch.rand(1).item()
            dec_input = top1
            if y is not None:
                dec_input = top1 if rand>RATIO else y[:, t]
            
        return outputs

In [ ]:
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
LR = 0.001
model = Translator('weights/eng_vectors.pt', 'weights/fr_vectors.pt', 50, len(word2index_fr), 64)
model.load_state_dict(torch.load('enc_dec_model.pt'))
model = model.to(DEVICE)

loss_fn = nn.CrossEntropyLoss()

In [ ]:
def trainer(model, dataloader, EPOCHS, pad_idx=0):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # CRITICAL: Tell the loss function to completely ignore <PAD> tokens
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx)
    
    model.train()
    
    for epoch in range(EPOCHS):
        net_loss = 0
        net_acc = 0
        loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
        
        for eng, fr in loop:
            eng = eng.to(DEVICE)
            fr = fr.to(DEVICE)
            
            optimizer.zero_grad()
            
            # 1. Pass the target sequence so the loop knows the max_len
            preds = model(eng, fr)
            
            # 2. Slice off the <SOS> token from both predictions and targets
            # preds shape becomes: (batch_size, 39, vocab_size)
            # fr shape becomes: (batch_size, 39)
            preds_sliced = preds[:, 1:]
            fr_sliced = fr[:, 1:]
            
            # 3. Flatten the tensors for CrossEntropyLoss
            # preds_flat shape: (batch_size * 39, vocab_size)
            preds_flat = preds_sliced.reshape(-1, preds_sliced.shape[-1])
            # fr_flat shape: (batch_size * 39)
            fr_flat = fr_sliced.reshape(-1)
            
            # Calculate loss (automatically ignores pad_idx)
            loss = loss_fn(preds_flat, fr_flat)
            
            loss.backward()
            
            # Optional but highly recommended for LSTMs: Gradient Clipping
            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            net_loss += loss.item()

            # 4. Calculate accurate accuracy (ignoring padding)
            pred_labels = torch.argmax(preds_sliced, dim=2)
            
            # Create a mask of True/False where the target is NOT a pad token
            non_pad_elements = (fr_sliced != pad_idx)
            
            # Count correct predictions only where it's not a pad token
            correct = ((pred_labels == fr_sliced) & non_pad_elements).sum().item()
            total_words = non_pad_elements.sum().item()
            
            acc = (correct / total_words) * 100 if total_words > 0 else 0
            net_acc += acc
            
        net_loss /= len(dataloader)
        net_acc /= len(dataloader)

        print(f"Loss = {net_loss:.5f} | Accuracy = {net_acc:.3f}%")
        torch.save(model.state_dict(), 'weights/enc_dec_model.pt')

In [15]:
trainer(model, dataloader, 2)

Loss = 3.19377 | Accuracy = 43.359%


Loss = 2.97175 | Accuracy = 45.236%


In [11]:
txt = "how are you"

preprocessed = nlp_eng(txt)
sentence = [word2index_eng[token.text.lower()] for token in preprocessed if not token.is_punct and not token.is_space]
print(sentence)
sentence = torch.tensor(sentence).unsqueeze(0)
with torch.inference_mode():
    sentence = sentence.to(DEVICE)
    pred = model(sentence, )
    pred = torch.squeeze(pred)
    pred = torch.argmax(pred, dim=1)
    for idx in pred:
        idx = idx.item()
        print(index2word_fr[idx])
        if(idx==3):
            break

[93, 465, 134]
<PAD>
comment
tu
l'
a
<EOS>


In [21]:
def tensor2text(indices_tensor, index2word, sos_idx, eos_idx, pad_idx):
    words = []
    for idx in indices_tensor:
        idx = idx.item() # Extract the Python integer from the PyTorch tensor
        
        if idx == eos_idx:
            break # Stop generating the sentence immediately
            
        if idx not in [sos_idx, pad_idx]: # Ignore SOS and PAD
            words.append(index2word[idx])
            
    return words

from nltk.translate.bleu_score import sentence_bleu
bleu = 0
for eng, fr in dataloader:
    eng = eng.to(DEVICE)
    # Let's say you just ran a batch through your model
    # eng shape: (batch_size, 40)
    # fr shape: (batch_size, 40)
    with torch.inference_mode():
        preds = model(eng) # preds shape: (batch_size, 40, vocab_size)

    # 1. Convert predictions from probabilities to integer indices
    pred_indices = preds.argmax(dim=2) # Shape becomes (batch_size, 40)

    batch_bleu_score = 0
    sos_idx = 2
    eos_idx = 3
    pad_idx = 0

    # 2. Loop through each sentence in the batch one by one
    for i in range(eng.shape[0]):
        
        # Extract the i-th predicted sentence and i-th target sentence
        pred_sentence_tensor = pred_indices[i] 
        target_sentence_tensor = fr[i]
        
        # 3. Convert both tensors back to clean lists of words
        candidate = tensor2text(pred_sentence_tensor, index2word_fr, sos_idx, eos_idx, pad_idx)
        target_words = tensor2text(target_sentence_tensor, index2word_fr, sos_idx, eos_idx, pad_idx)
        
        # 4. Format the reference for BLEU
        # NLTK expects the reference to be a list of lists: [[word1, word2, ...]]
        reference = [target_words] 
        
        # 5. Calculate the score for this specific sentence
        # We use a try-except block because very short, completely wrong 
        # predictions can sometimes cause a math error in NLTK's BLEU calculator
        try:
            score = sentence_bleu(reference, candidate)
        except Exception:
            score = 0.0
            
        batch_bleu_score += score

    # Get the average BLEU for the batch
    average_batch_bleu = 100*batch_bleu_score / eng.shape[0]
    bleu += average_batch_bleu
bleu /= len(dataloader)
print(bleu)

/Users/vatsalvarenya/Developer /python/nlp/eng2french/.venv/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/Users/vatsalvarenya/Developer /python/nlp/eng2french/.venv/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/Users/vatsalvarenya/Developer /python/nlp/eng2french/.venv/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, i

9.225007579146881
